In [17]:
import pandas as pd
import numpy as np
from glob import glob
import re

from fontTools.diff import color
from pandas.core.array_algos import replace

In [19]:
# Map of the thresholds per district
# TODO match the district names on the threshold df (but not use that new df for other purposes)

thresholds = pd.read_csv('/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Outputs/thresholds_custom_year_infull.csv')
pop = pd.read_csv("~/Desktop/MSC THESIS/Data works/Code/Outputs/population.csv")


In [20]:
thresholds.columns


Index(['Unnamed: 0', 'STATE', 'DISTRICT', 'THRESHOLD_10YR'], dtype='str')

In [21]:
import re
import unicodedata

def clean_name(x):
    if pd.isna(x):
        return x

    # Normalize unicode (fix hidden encoding issues)
    x = unicodedata.normalize("NFKD", x)

    # Remove non-ASCII characters
    x = x.encode("ascii", "ignore").decode("ascii")

    # Uppercase
    x = x.upper()

    # Remove special characters (like >, |, @)
    x = re.sub(r"[^A-Z0-9 ]", "", x)

    # Remove extra spaces
    x = re.sub(r"\s+", " ", x).strip()

    return x

In [22]:
pop["DISTRICT_CLEAN"] = pop["DISTRICT"].apply(clean_name)
thresholds["DISTRICT_CLEAN"] = thresholds["DISTRICT"].apply(clean_name)

In [23]:
thresholds['DISTRICT'].nunique()

770

In [24]:
# OPTIONAL, might create errors
from rapidfuzz import process

choices = pop["DISTRICT_CLEAN"].unique()

def fuzzy_match(x):
    match, score, _ = process.extractOne(x, choices)
    return match if score > 90 else None


In [25]:
thresholds["DISTRICT_MATCH"] = thresholds["DISTRICT_CLEAN"].apply(fuzzy_match)

In [26]:
thresholds.to_csv("/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Outputs/final material/clean_thresholds.csv", index=False)